# EEG Single Instance Overfitting Verification

This notebook verifies that the model successfully overfits a single EEG segment.

**Configuration:**
- EEG shape: (1, 2, 600) - 1 channel, 2 electrodes, 600 time samples
- Patch size: (2, 10) - 60 total patches

**Expected behavior:**
- Generated signals should closely match the target
- MSE between generated and target should be very low (<0.01)
- Visual comparison should show near-identical waveforms

## Setup

### Training command:
```bash
python train_eeg_single.py --max_steps 5000
```

Or with your own EEG data:
```bash
python train_eeg_single.py --data_path path/to/eeg.npy --max_steps 5000
```

In [ ]:
# Navigate to PixNerd folder
import os
import sys

NOTEBOOK_DIR = os.getcwd()
print(f"Starting directory: {NOTEBOOK_DIR}")

# Navigate to PixNerd folder (where src/ lives)
PIXNERD_DIR = os.path.join(NOTEBOOK_DIR, "PixNerd")
if os.path.exists(PIXNERD_DIR):
    os.chdir(PIXNERD_DIR)
    print(f"Changed to: {os.getcwd()}")
elif os.path.basename(NOTEBOOK_DIR) == "PixNerd":
    print(f"Already in PixNerd directory: {NOTEBOOK_DIR}")
else:
    parent = os.path.dirname(NOTEBOOK_DIR)
    pixnerd_in_parent = os.path.join(parent, "PixNerd")
    if os.path.exists(pixnerd_in_parent):
        os.chdir(pixnerd_in_parent)
        print(f"Changed to: {os.getcwd()}")
    else:
        print(f"WARNING: Could not find PixNerd folder")

if os.path.exists("src"):
    print("Found src/ directory")
else:
    print("ERROR: src/ directory not found!")

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Paths
PIXNERD_ROOT = Path(os.getcwd())

# ============================================================
# CHECKPOINT PATH - UPDATE THIS TO YOUR TRAINED MODEL
# ============================================================
EXP_DIR = PIXNERD_ROOT / "workdirs" / "exp_eeg_single_overfit"
CKPT_PATH = EXP_DIR / "checkpoints" / "last.ckpt"
TARGET_SIGNAL_PATH = EXP_DIR / "target_signal.npy"
CONFIG_PATH = EXP_DIR / "config.json"
# ============================================================

# ============================================================
# WEIGHT SELECTION - CRITICAL FOR OVERFITTING VERIFICATION
# ============================================================
# Training loss is computed on the ONLINE denoiser (not EMA).
# For overfitting verification, set USE_EMA_WEIGHTS = False
# to load the same weights that achieved the low training loss.
#
# USE_EMA_WEIGHTS = True  -> Load EMA denoiser (smoother, but lags behind)
# USE_EMA_WEIGHTS = False -> Load online denoiser (matches training loss)
# ============================================================
USE_EMA_WEIGHTS = False  # Set to False for overfitting tests!
# ============================================================

OUTPUT_DIR = PIXNERD_ROOT / "outputs" / "eeg_single"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Checkpoint path: {CKPT_PATH}")
print(f"Checkpoint exists: {CKPT_PATH.exists()}")
print(f"Target signal exists: {TARGET_SIGNAL_PATH.exists()}")
print(f"Config exists: {CONFIG_PATH.exists()}")
print(f"Device: {DEVICE}")
print()
print(f"*** Using {'EMA' if USE_EMA_WEIGHTS else 'ONLINE'} denoiser weights ***")
if not USE_EMA_WEIGHTS:
    print("    (Online weights match training loss computation)")

## Load Configuration and Target Signal

In [ ]:
# Load config
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        config = json.load(f)
    print("Loaded config:")
    for k, v in config.items():
        print(f"  {k}: {v}")
else:
    # Default config
    config = {
        "num_channels": 2,
        "num_samples": 600,
        "patch_size_h": 2,
        "patch_size_w": 10,
        "hidden_size": 256,
        "decoder_hidden_size": 32,
        "num_encoder_blocks": 6,
        "num_decoder_blocks": 2,
        "num_groups": 8,
    }
    print("Using default config")

# Extract config
NUM_CHANNELS = config["num_channels"]
NUM_SAMPLES = config["num_samples"]
PATCH_SIZE_H = config["patch_size_h"]
PATCH_SIZE_W = config["patch_size_w"]
HIDDEN_SIZE = config["hidden_size"]
DECODER_HIDDEN_SIZE = config["decoder_hidden_size"]
NUM_ENCODER_BLOCKS = config["num_encoder_blocks"]
NUM_DECODER_BLOCKS = config["num_decoder_blocks"]
NUM_GROUPS = config["num_groups"]

num_patches = (NUM_CHANNELS // PATCH_SIZE_H) * (NUM_SAMPLES // PATCH_SIZE_W)
print(f"\nTotal patches: {num_patches}")

In [ ]:
# Load target signal
target_signal = np.load(TARGET_SIGNAL_PATH)
target_tensor = torch.from_numpy(target_signal).float()

print(f"Target signal shape: {target_signal.shape}")
print(f"Target range: [{target_signal.min():.3f}, {target_signal.max():.3f}]")

# Visualize target signal
fig, axes = plt.subplots(NUM_CHANNELS, 1, figsize=(14, 3 * NUM_CHANNELS))
if NUM_CHANNELS == 1:
    axes = [axes]

time = np.arange(NUM_SAMPLES) / 200.0  # Assuming 200 Hz sampling rate

for ch in range(NUM_CHANNELS):
    axes[ch].plot(time, target_signal[0, ch, :], 'b-', linewidth=0.8)
    axes[ch].set_ylabel(f"Channel {ch}")
    axes[ch].set_xlabel("Time (s)")
    axes[ch].set_xlim([0, time[-1]])
    axes[ch].grid(True, alpha=0.3)

plt.suptitle("Target EEG Signal (to be memorized)", fontsize=14)
plt.tight_layout()
plt.show()

## Build Model

In [ ]:
# Import PixNerd components
from src.models.autoencoder.pixel import PixelAE
from src.models.autoencoder.base import fp2uint8
from src.models.conditioner.class_label import LabelConditioner
from src.models.transformer.pixnerd_eeg_heavydecoder import PixNerDiT_EEG
from src.diffusion.flow_matching.scheduling import LinearScheduler
from src.diffusion.flow_matching.sampling import EulerSampler, ode_step_fn
from src.diffusion.base.guidance import simple_guidance_fn

print("Imports successful!")

In [ ]:
print("Initializing model components...")

main_scheduler = LinearScheduler()

vae = PixelAE(scale=1.0)

conditioner = LabelConditioner(num_classes=1)

# Create denoiser directly (we'll load EMA weights into this)
denoiser = PixNerDiT_EEG(
    in_channels=1,
    patch_size_h=PATCH_SIZE_H,
    patch_size_w=PATCH_SIZE_W,
    num_groups=NUM_GROUPS,
    hidden_size=HIDDEN_SIZE,
    decoder_hidden_size=DECODER_HIDDEN_SIZE,
    num_encoder_blocks=NUM_ENCODER_BLOCKS,
    num_decoder_blocks=NUM_DECODER_BLOCKS,
    num_classes=1,
)

# Sampler with no guidance for overfitting
sampler = EulerSampler(
    num_steps=50,
    guidance=1.0,
    guidance_interval_min=0.0,
    guidance_interval_max=1.0,
    scheduler=main_scheduler,
    w_scheduler=LinearScheduler(),
    guidance_fn=simple_guidance_fn,
    step_fn=ode_step_fn,
)

print(f"Denoiser parameters: {sum(p.numel() for p in denoiser.parameters()):,}")
print("Model components initialized!")

## Load Checkpoint

In [ ]:
print(f"Loading checkpoint from: {CKPT_PATH}")
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)

# Analyze checkpoint structure
state_dict = ckpt["state_dict"]
print(f"\nCheckpoint keys: {len(state_dict)}")

# Count keys by prefix
prefixes = {}
for key in state_dict.keys():
    prefix = key.split('.')[0]
    prefixes[prefix] = prefixes.get(prefix, 0) + 1
print(f"Key prefixes: {prefixes}")

# ============================================================
# Compare EMA vs Online weights to understand the difference
# ============================================================
print("\n" + "="*60)
print("WEIGHT COMPARISON: EMA vs Online Denoiser")
print("="*60)

ema_prefix = "ema_denoiser."
online_prefix = "denoiser."

# Get sample weights for comparison
sample_key_suffix = None
for key in state_dict.keys():
    if key.startswith(ema_prefix) and "weight" in key and "norm" in key:
        sample_key_suffix = key[len(ema_prefix):]
        break

if sample_key_suffix:
    ema_key = ema_prefix + sample_key_suffix
    online_key = online_prefix + sample_key_suffix
    
    if ema_key in state_dict and online_key in state_dict:
        ema_weight = state_dict[ema_key]
        online_weight = state_dict[online_key]
        
        weight_diff = (ema_weight - online_weight).abs()
        print(f"\nSample weight: {sample_key_suffix}")
        print(f"  EMA weight stats:    mean={ema_weight.mean():.6f}, std={ema_weight.std():.6f}")
        print(f"  Online weight stats: mean={online_weight.mean():.6f}, std={online_weight.std():.6f}")
        print(f"  Difference: mean_abs_diff={weight_diff.mean():.6f}, max_abs_diff={weight_diff.max():.6f}")
        
        if weight_diff.max() < 1e-6:
            print("  -> Weights are nearly IDENTICAL (EMA has fully converged)")
        elif weight_diff.mean() < 0.01:
            print("  -> Weights are SIMILAR (EMA is close to online)")
        else:
            print("  -> Weights are DIFFERENT (EMA lags behind online)")
            print("     For overfitting test, USE_EMA_WEIGHTS=False is recommended!")

print("="*60)

# ============================================================
# Load weights based on USE_EMA_WEIGHTS setting
# ============================================================
if USE_EMA_WEIGHTS:
    weight_prefix = ema_prefix
    print(f"\nLoading EMA denoiser weights (prefix: '{weight_prefix}')...")
else:
    weight_prefix = online_prefix
    print(f"\nLoading ONLINE denoiser weights (prefix: '{weight_prefix}')...")

selected_weights = {}
for key, value in state_dict.items():
    if key.startswith(weight_prefix):
        new_key = key[len(weight_prefix):]
        selected_weights[new_key] = value

print(f"Found {len(selected_weights)} denoiser keys")

# Check if keys match
denoiser_keys = set(denoiser.state_dict().keys())
selected_keys = set(selected_weights.keys())

missing = denoiser_keys - selected_keys
unexpected = selected_keys - denoiser_keys

print(f"Missing in checkpoint: {len(missing)}")
print(f"Unexpected in checkpoint: {len(unexpected)}")

if missing:
    print(f"  Missing keys: {list(missing)[:5]}...")
if unexpected:
    print(f"  Unexpected keys: {list(unexpected)[:5]}...")

# Load the weights
result = denoiser.load_state_dict(selected_weights, strict=False)
print(f"\nload_state_dict result:")
print(f"  Missing: {len(result.missing_keys)}")
print(f"  Unexpected: {len(result.unexpected_keys)}")

# Move to device and ensure float32 (EMA is kept in float32 during training)
denoiser.to(DEVICE)
denoiser.float()  # Explicit float32
denoiser.eval()

# Verify model precision
sample_param = next(denoiser.parameters())
print(f"\nModel precision: {sample_param.dtype}")
print(f"Model device: {sample_param.device}")

# Verify weights are loaded (check a sample weight)
print("\nVerifying loaded weights:")
if hasattr(denoiser, 'blocks') and len(denoiser.blocks) > 0:
    sample_weight = denoiser.blocks[0].norm1.weight
    print(f"  blocks[0].norm1.weight: shape={sample_weight.shape}")
    print(f"    mean={sample_weight.mean().item():.6f}")
    print(f"    std={sample_weight.std().item():.6f}")
    print(f"    min={sample_weight.min().item():.6f}")
    print(f"    max={sample_weight.max().item():.6f}")
    
    # For RMSNorm, default weight is all 1s - check if we loaded different values
    if torch.allclose(sample_weight, torch.ones_like(sample_weight), atol=1e-4):
        print("    WARNING: Weight appears to be at initialization (all 1s)!")
    else:
        print("    Weights appear to be loaded correctly (not default values)")

# Also check class embedding
if hasattr(denoiser, 'y_embedder'):
    class_weight = denoiser.y_embedder.embedding_table.weight
    print(f"\n  y_embedder.embedding_table.weight: shape={class_weight.shape}")
    print(f"    mean={class_weight.mean().item():.6f}")
    print(f"    std={class_weight.std().item():.6f}")
    print(f"    Expected shape: [2, {HIDDEN_SIZE}] (1 class + 1 null)")

print("\nCheckpoint loaded successfully!")
print(f"*** Loaded {'EMA' if USE_EMA_WEIGHTS else 'ONLINE'} weights ***")

# Quick forward pass test
print("\nTesting forward pass...")
with torch.no_grad():
    test_noise = torch.randn(1, 1, NUM_CHANNELS, NUM_SAMPLES, device=DEVICE)
    test_t = torch.tensor([0.5], device=DEVICE)
    test_class = torch.tensor([0], device=DEVICE)
    
    try:
        test_output = denoiser(test_noise, test_t, test_class)
        print(f"  Input shape: {test_noise.shape}")
        print(f"  Output shape: {test_output.shape}")
        print(f"  Output range: [{test_output.min().item():.3f}, {test_output.max().item():.3f}]")
        print("  Forward pass successful!")
    except Exception as e:
        print(f"  Forward pass failed: {e}")

## Helper Functions

In [ ]:
@torch.no_grad()
def sample_eeg(
    num_samples: int = 1,
    seed: int = 42,
    num_steps: int = 50,
    guidance: float = 1.0,
    use_autocast: bool = True,
):
    """Generate EEG samples.
    
    Args:
        num_samples: Number of samples to generate
        seed: Random seed for reproducibility
        num_steps: Number of sampling steps
        guidance: CFG guidance scale (1.0 = no guidance)
        use_autocast: Whether to use bfloat16 autocast (default True, set False for debugging)
    """
    torch.manual_seed(seed)
    
    # Configure sampler
    sampler.guidance = guidance
    sampler.num_steps = num_steps
    
    # Generate noise with correct shape
    noise = torch.randn(num_samples, 1, NUM_CHANNELS, NUM_SAMPLES, device=DEVICE)
    
    # Get condition (class 0)
    labels = [0] * num_samples
    condition, uncondition = conditioner(labels)
    condition = condition.to(DEVICE)
    uncondition = uncondition.to(DEVICE)
    
    # Sample using standalone denoiser
    # NOTE: The sampler has @torch.autocast("cuda", dtype=torch.bfloat16) decorator
    # which may cause precision issues. For debugging, we can bypass this.
    if use_autocast:
        samples = sampler(
            denoiser,
            noise,
            condition,
            uncondition,
        )
    else:
        # Manual sampling without autocast for debugging precision issues
        with torch.cuda.amp.autocast(enabled=False):
            samples = sampler._impl_sampling(denoiser, noise.float(), condition, uncondition)[0][-1]
    
    # Decode (identity for PixelAE)
    signals = vae.decode(samples)
    signals = torch.clamp(signals, -1.0, 1.0)
    
    return signals.cpu()


def compute_metrics(generated, target):
    """Compute MSE and correlation between generated and target signals."""
    mse = F.mse_loss(generated, target).item()
    
    # Flatten and compute correlation
    gen_flat = generated.flatten().numpy()
    tgt_flat = target.flatten().numpy()
    correlation = np.corrcoef(gen_flat, tgt_flat)[0, 1]
    
    return mse, correlation


def plot_comparison(target, generated, title=""):
    """Plot comparison between target and generated signals."""
    time = np.arange(NUM_SAMPLES) / 200.0
    
    fig, axes = plt.subplots(NUM_CHANNELS, 1, figsize=(14, 3 * NUM_CHANNELS))
    if NUM_CHANNELS == 1:
        axes = [axes]
    
    for ch in range(NUM_CHANNELS):
        axes[ch].plot(time, target[0, ch, :], 'b-', linewidth=1.0, label='Target', alpha=0.7)
        axes[ch].plot(time, generated[0, ch, :], 'r--', linewidth=1.0, label='Generated', alpha=0.7)
        axes[ch].set_ylabel(f"Channel {ch}")
        axes[ch].set_xlabel("Time (s)")
        axes[ch].set_xlim([0, time[-1]])
        axes[ch].legend(loc='upper right')
        axes[ch].grid(True, alpha=0.3)
    
    if title:
        plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    return fig


@torch.no_grad()
def compute_training_loss_at_t(target_signal, t_value=0.5):
    """
    Compute what the training loss would be for the target signal at timestep t.
    This helps verify the model's behavior matches training.
    """
    # Prepare inputs like training
    x = target_signal.unsqueeze(0).to(DEVICE)  # [1, 1, 2, 600]
    t = torch.tensor([t_value], device=DEVICE)
    y = torch.tensor([0], device=DEVICE)  # Class 0
    
    # Create noise
    noise = torch.randn_like(x)
    
    # Compute alpha, sigma for LinearScheduler
    alpha = t.view(-1, 1, 1, 1)  # t
    sigma = (1 - t).view(-1, 1, 1, 1)  # 1-t
    
    # Create noisy sample: x_t = alpha * x + sigma * noise
    x_t = alpha * x + sigma * noise
    
    # Target velocity: v_t = dalpha * x + dsigma * noise = x - noise
    v_t = x - noise
    
    # Get model prediction
    v_pred = denoiser(x_t, t, y)
    
    # Compute loss
    loss = F.mse_loss(v_pred, v_t)
    
    return loss.item(), x_t, v_t, v_pred


print("Helper functions defined.")
print("\nAdditional diagnostic function: compute_training_loss_at_t()")
print("  - This verifies the model's forward pass matches training expectations")

In [ ]:
# Diagnostic: Verify training loss computation at inference time
# If the model truly memorized the signal, the training loss should be very low

print("="*60)
print("DIAGNOSTIC: Verifying training loss at inference time")
print(f"Using {'EMA' if USE_EMA_WEIGHTS else 'ONLINE'} denoiser weights")
print("="*60)

# Test at multiple timesteps
timesteps_to_test = [0.1, 0.3, 0.5, 0.7, 0.9]
losses = []

print(f"\nComputing training loss for EEG signal:")
for t_val in timesteps_to_test:
    torch.manual_seed(42)  # Fixed seed for reproducibility
    loss, x_t, v_t, v_pred = compute_training_loss_at_t(target_tensor, t_val)
    losses.append(loss)
    print(f"  t={t_val:.1f}: loss={loss:.6f}")

avg_loss = np.mean(losses)
print(f"\nAverage loss across timesteps: {avg_loss:.6f}")

if avg_loss < 0.001:
    print("EXCELLENT: Model has very low training loss - weights are correctly loaded!")
    print("           This matches the training loss reported during training.")
elif avg_loss < 0.01:
    print("GOOD: Model has low training loss - should produce good reconstructions.")
elif avg_loss < 0.1:
    print("WARNING: Model has moderate loss - may not have fully memorized the signal.")
    if USE_EMA_WEIGHTS:
        print("         TRY: Set USE_EMA_WEIGHTS = False to use online denoiser weights.")
else:
    print("ERROR: Model has high loss - weights may not be correctly loaded!")
    if USE_EMA_WEIGHTS:
        print("       TRY: Set USE_EMA_WEIGHTS = False to use online denoiser weights.")
    else:
        print("       Check that training completed successfully.")

print("="*60)

## Generate and Compare

Generate samples with different seeds and compare to target.

In [ ]:
# Generate multiple samples - compare autocast vs full precision
num_test_samples = 5
all_samples = []
all_mses = []
all_corrs = []
all_samples_nocast = []
all_mses_nocast = []
all_corrs_nocast = []

print("Generating samples with autocast (bfloat16):")
for seed in range(num_test_samples):
    samples = sample_eeg(
        num_samples=1,
        seed=seed,
        num_steps=100,
        guidance=1.0,
        use_autocast=True,
    )
    all_samples.append(samples[0])
    
    # Compute metrics
    mse, corr = compute_metrics(samples[0], target_tensor)
    all_mses.append(mse)
    all_corrs.append(corr)
    print(f"  Seed {seed}: MSE={mse:.6f}, Correlation={corr:.4f}")

print(f"\nAutocast - Average MSE: {np.mean(all_mses):.6f}, Average Correlation: {np.mean(all_corrs):.4f}")

# Also test without autocast for comparison (if on CUDA)
if DEVICE == "cuda":
    print("\nGenerating samples WITHOUT autocast (float32):")
    for seed in range(num_test_samples):
        samples = sample_eeg(
            num_samples=1,
            seed=seed,
            num_steps=100,
            guidance=1.0,
            use_autocast=False,
        )
        all_samples_nocast.append(samples[0])
        
        mse, corr = compute_metrics(samples[0], target_tensor)
        all_mses_nocast.append(mse)
        all_corrs_nocast.append(corr)
        print(f"  Seed {seed}: MSE={mse:.6f}, Correlation={corr:.4f}")
    
    print(f"\nNo autocast - Average MSE: {np.mean(all_mses_nocast):.6f}, Average Correlation: {np.mean(all_corrs_nocast):.4f}")
    
    # Compare
    print(f"\nPrecision impact: MSE diff = {np.mean(all_mses) - np.mean(all_mses_nocast):.6f}")
else:
    print("\n(Skipping no-autocast comparison - not on CUDA)")
    all_samples_nocast = all_samples
    all_mses_nocast = all_mses
    all_corrs_nocast = all_corrs

In [ ]:
# Visualize best sample
best_idx = np.argmin(all_mses)
best_sample = all_samples[best_idx]

fig = plot_comparison(
    target_signal,
    best_sample.numpy(),
    title=f"Best Sample (seed={best_idx}, MSE={all_mses[best_idx]:.6f}, r={all_corrs[best_idx]:.4f})"
)
plt.savefig(OUTPUT_DIR / "overfitting_comparison.png", dpi=150)
plt.show()

In [ ]:
# Visualize all samples overlaid
time = np.arange(NUM_SAMPLES) / 200.0

fig, axes = plt.subplots(NUM_CHANNELS, 1, figsize=(14, 3 * NUM_CHANNELS))
if NUM_CHANNELS == 1:
    axes = [axes]

for ch in range(NUM_CHANNELS):
    # Plot target
    axes[ch].plot(time, target_signal[0, ch, :], 'k-', linewidth=2.0, label='Target', zorder=10)
    
    # Plot all generated samples
    for i, sample in enumerate(all_samples):
        sample_np = sample.numpy()
        label = f'Seed {i}' if i == 0 else None
        axes[ch].plot(time, sample_np[0, ch, :], '--', linewidth=0.8, alpha=0.5, label=f'Seed {i}')
    
    axes[ch].set_ylabel(f"Channel {ch}")
    axes[ch].set_xlabel("Time (s)")
    axes[ch].set_xlim([0, time[-1]])
    axes[ch].legend(loc='upper right', fontsize=8)
    axes[ch].grid(True, alpha=0.3)

plt.suptitle("All Generated Samples vs Target", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "all_samples_comparison.png", dpi=150)
plt.show()

## Difference Analysis

In [ ]:
# Analyze differences
best_sample_np = best_sample.numpy()
diff = np.abs(best_sample_np - target_signal)

fig, axes = plt.subplots(NUM_CHANNELS, 1, figsize=(14, 3 * NUM_CHANNELS))
if NUM_CHANNELS == 1:
    axes = [axes]

time = np.arange(NUM_SAMPLES) / 200.0

for ch in range(NUM_CHANNELS):
    axes[ch].fill_between(time, 0, diff[0, ch, :], alpha=0.7, color='red')
    axes[ch].set_ylabel(f"Channel {ch}\n|Error|")
    axes[ch].set_xlabel("Time (s)")
    axes[ch].set_xlim([0, time[-1]])
    axes[ch].grid(True, alpha=0.3)
    max_err = diff[0, ch, :].max()
    mean_err = diff[0, ch, :].mean()
    axes[ch].set_title(f"Max error: {max_err:.4f}, Mean error: {mean_err:.4f}")

plt.suptitle("Absolute Error (Best Sample)", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "error_analysis.png", dpi=150)
plt.show()

## Metrics Summary

In [ ]:
print("="*60)
print("EEG OVERFITTING VERIFICATION SUMMARY")
print("="*60)
print(f"Signal shape: (1, {NUM_CHANNELS}, {NUM_SAMPLES})")
print(f"Patch size: ({PATCH_SIZE_H}, {PATCH_SIZE_W})")
print(f"Total patches: {num_patches}")
print(f"Weight source: {'EMA' if USE_EMA_WEIGHTS else 'ONLINE'} denoiser")
print()
print(f"Number of samples tested: {num_test_samples}")
print(f"Average MSE: {np.mean(all_mses):.6f}")
print(f"Average Correlation: {np.mean(all_corrs):.4f}")
print(f"Best MSE: {np.min(all_mses):.6f} (seed {np.argmin(all_mses)})")
print(f"Best Correlation: {np.max(all_corrs):.4f} (seed {np.argmax(all_corrs)})")
print()

# Quality assessment
avg_mse = np.mean(all_mses)
avg_corr = np.mean(all_corrs)

if avg_mse < 0.001 and avg_corr > 0.99:
    print("EXCELLENT: Model has perfectly memorized the EEG signal!")
elif avg_mse < 0.01 and avg_corr > 0.95:
    print("GOOD: Model has mostly memorized the EEG signal.")
elif avg_mse < 0.05 and avg_corr > 0.9:
    print("FAIR: Model is learning but needs more training.")
    if USE_EMA_WEIGHTS:
        print("      TRY: Set USE_EMA_WEIGHTS = False to test with online weights.")
else:
    print("POOR: Model has not yet memorized the signal.")
    if USE_EMA_WEIGHTS:
        print("      TRY: Set USE_EMA_WEIGHTS = False to test with online weights.")
    else:
        print("      Consider training for more steps.")

print("="*60)

## Time-Domain Super-Resolution Test (Optional)

Test if the overfitted model can generate at higher temporal resolution.

In [ ]:
def set_decoder_scale(scale_h: float, scale_w: float):
    """Set decoder patch scaling for super-resolution."""
    denoiser.decoder_patch_scaling_h = scale_h
    denoiser.decoder_patch_scaling_w = scale_w


@torch.no_grad()
def sample_superres(
    height: int,
    width: int,
    seed: int = 42,
    num_steps: int = 50,
):
    """Generate super-resolution sample."""
    torch.manual_seed(seed)
    
    scale_h = height / NUM_CHANNELS
    scale_w = width / NUM_SAMPLES
    set_decoder_scale(scale_h, scale_w)
    
    sampler.guidance = 1.0
    sampler.num_steps = num_steps
    
    noise = torch.randn(1, 1, height, width, device=DEVICE)
    
    condition, uncondition = conditioner([0])
    condition = condition.to(DEVICE)
    uncondition = uncondition.to(DEVICE)
    
    samples = sampler(
        denoiser,
        noise,
        condition,
        uncondition,
    )
    
    signals = vae.decode(samples)
    signals = torch.clamp(signals, -1.0, 1.0)
    
    # Reset scale
    set_decoder_scale(1.0, 1.0)
    
    return signals.cpu()


# Generate at different time resolutions (keep channels same)
print("Generating time-domain super-resolution samples...")

sig_native = best_sample  # Already have this
sig_2x = sample_superres(NUM_CHANNELS, NUM_SAMPLES * 2, seed=best_idx, num_steps=100)  # 1200 samples

# Compare native vs 2x resolution
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

time_native = np.arange(NUM_SAMPLES) / 200.0
time_2x = np.arange(NUM_SAMPLES * 2) / 400.0  # 2x sampling rate

# Native resolution
axes[0].plot(time_native, sig_native[0, 0, :].numpy(), 'b-', linewidth=1.0)
axes[0].set_title(f"Native Resolution ({NUM_SAMPLES} samples)")
axes[0].set_xlabel("Time (s)")
axes[0].set_xlim([0, max(time_native[-1], time_2x[-1])])
axes[0].grid(True, alpha=0.3)

# 2x resolution
axes[1].plot(time_2x, sig_2x[0, 0, 0, :].numpy(), 'r-', linewidth=1.0)
axes[1].set_title(f"2x Super-Resolution ({NUM_SAMPLES * 2} samples)")
axes[1].set_xlabel("Time (s)")
axes[1].set_xlim([0, max(time_native[-1], time_2x[-1])])
axes[1].grid(True, alpha=0.3)

plt.suptitle("Time-Domain Super-Resolution", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "superres_comparison.png", dpi=150)
plt.show()

In [ ]:
print("Done!")
print(f"Outputs saved to: {OUTPUT_DIR}")